In [12]:
# install and importing the gpu for training the model fast
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Food Image Classification and Nutrition Estimation System

##  Project Overview

This project uses Deep Learning and Transfer Learning to classify food images into 11 categories using MobileNetV2.  

After predicting the food item, the system provides:
- Nutritional information (calories and protein per 100g)
- Nutrition calculation based on user-entered food weight
- Personalized daily calorie and protein requirements based on body weight

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### image preprocessing
1. th imagedatagenerator from the tensorflow and it is rescaled into 0 and so tht it will be easily understand for cnn
2. random rotate 25"c so that the training will be easy
3. randomly zoom and the image will be sligthly tiled also all are used to overcome overfitting

In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.25,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(rescale=1./255)


In [ ]:

!cp -ru "/content/drive/MyDrive/ cnn food" "/content/"

In [ ]:

!ls /content/


' cnn food'   drive   sample_data


##loading all the packages and library

In [15]:

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import os
from tensorflow.keras.regularizers import l2

In [ ]:
!ls -l "/content/drive/MyDrive"



total 18514
-rw------- 1 root root 17390560 Feb 21 10:02  best_food_model.h5
-rw------- 1 root root     6010 Feb 21 10:18  bread.jpg
-rw------- 1 root root   120045 Feb 21 10:26  Butter-Noodles-S5.jpg
drwx------ 5 root root     4096 Feb 17 14:43 ' cnn food'
drwx------ 2 root root     4096 Feb 17 13:34 'Colab Notebooks'
-rw------- 1 root root     7800 Feb 21 10:18  download.jpg
-rw------- 1 root root    10778 Feb 21 10:18 'images (1).jfif'
-rw------- 1 root root     7157 Feb 21 10:29 'images (2).jfif'
-rw------- 1 root root    16953 Feb 21 10:29 'images (3).jfif'
-rw------- 1 root root   729192 Feb 21 10:28  iStock-544807136.jpg
-rw------- 1 root root   632397 Feb 21 10:25  Popular-Fried-Foods.jpg
-rw------- 1 root root     8219 Feb 21 10:19  rice.jpg
-rw------- 1 root root    17222 Feb 21 10:25  sea.jpg


##Loading Training Data

1.  We are loading the **training dataset path**.
2. Each folder inside the training directory represents a class label(0, 1, 2).
3. A data generator is created to:
  - Load images in batches
  - Feed them into the model during training
- Each image is resized to 224 × 224 pixels
- The batch size is set to 64
4. This means **64 images are processed per batch**

In [16]:


train_path = "/content/drive/MyDrive/ cnn food/train"
val_path = "/content/drive/MyDrive/ cnn food/validation"

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical"
)



Found 9869 images belonging to 11 classes.


###Validation Data Preparation

1. We are loading the validation dataset path.
2. Each folder inside the validation directory represents a class label (0, 1, 2).
3. A data generator is created to:
  - Load images in batches
  - Feed them into the model for validation
4. Each image is resized to 224 × 224 pixels
 - The batch size is set to 64
- This means 64 images are processed per batch
5. The images are **not shuffled**
  - This ensures consistent evaluation results

In [17]:
val_generator = val_datagen.flow_from_directory(
    val_path,
    target_size=(224, 224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

Found 3434 images belonging to 11 classes.


## MobileNetV2 Model

1. Loading the pretrained **MobileNetV2** model from TensorFlow (developed by Google).
2. Pretrained weights are loaded from ImageNet.
3. include_top=False removes the original 1000-class output layer.
4. Our model requires 11 output classes.
5. input_shape defines image height, width (224 × 224), and 3 RGB channels.

In [1]:

from tensorflow.keras.applications import MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


## CNN Model Building

1. Imported required libraries for CNN model development.
2. Sequential builds the model layer by layer.
3. Dense represents fully connected layers.
4. Dropout randomly disables neurons to reduce overfitting.
5. GlobalAveragePooling converts feature maps into a single vector (better than Flatten).
6. BatchNormalization normalizes values between layers.
7. Softmax is used in the output layer with 11 neurons to predict class probabilities.

In [ ]:

from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import BatchNormalization
model = Sequential([
    base_model,

    GlobalAveragePooling2D(),

    BatchNormalization(),

    Dense(512, activation="relu"),

    Dropout(0.5),

    Dense(train_generator.num_classes, activation="softmax")
])


## Model Compilation

1. The model is compiled in this step.
2. Adam optimizer adjusts the weights during training.
3. The learning rate controls how fast the model learns (lower loss indicates better performance).
4. Accuracy is used as a metric to measure how many predictions are correct.

In [ ]:

from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

##Early Stopping

1. Prevents unnecessary training.
2. Monitors validation accuracy during training.
3. Waits for 5 epochs to check performance.
4. Stops training if there is no improvement.

In [ ]:

from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True
)


##Reduce Learning Rate Callback

1. ReduceLROnPlateau is imported from TensorFlow callbacks.
2. It automatically reduces the learning rate when validation loss stops improving.
3. The learning rate is multiplied by 0.3.
4.  It waits for 3 epochs before reducing.
5. **bold text** The minimum learning rate is set to 1e-6.

In [ ]:

from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

## Model Checkpoint Callback

1. ModelCheckpoint is a callback used to save the best version of the model during training.
2. It monitors validation accuracy.
3. save_best_only=True ensures only the best-performing model is saved.
4. The model is stored safely at the specified path in Google Drive.

In [ ]:

from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/best_food_model.h5",
    monitor="val_accuracy",
    save_best_only=True
)

## Model Training

1. Training starts in this step.
2. Validation data is used to check model performance and detect overfitting.
3. epochs define the number of training cycles.
4. Each epoch contains 155 batches (Total Images ÷ Batch Size).
5. More epochs can improve performance (if not overfitting).
6. Final achieved accuracy: **87%**.

In [ ]:

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stop, reduce_lr, checkpoint]
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3767 - loss: 2.0729

155/155 ━━━━━━━━━━━━━━━━━━━━ 237s 1s/step - accuracy: 0.3775 - loss: 2.0700 - val_accuracy: 0.7312 - val_loss: 0.8675 - learning_rate: 1.0000e-04
Epoch 2/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6646 - loss: 1.0547

155/155 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.6647 - loss: 1.0545 - val_accuracy: 0.7732 - val_loss: 0.7088 - learning_rate: 1.0000e-04
Epoch 3/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7132 - loss: 0.8871

155/155 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.7133 - loss: 0.8869 - val_accuracy: 0.7953 - val_loss: 0.6608 - learning_rate: 1.0000e-04
Epoch 4/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7450 - loss: 0.7774

...

155/155 ━━━━━━━━━━━━━━━━━━━━ 183s 1s/step - accuracy: 0.8527 - loss: 0.4454 - val_accuracy: 0.8363 - val_loss: 0.5069 - learning_rate: 1.0000e-04
Epoch 19/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8565 - loss: 0.4178

155/155 ━━━━━━━━━━━━━━━━━━━━ 183s 1s/step - accuracy: 0.8565 - loss: 0.4179 - val_accuracy: 0.8369 - val_loss: 0.5050 - learning_rate: 1.0000e-04
Epoch 20/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.8606 - loss: 0.4003 - val_accuracy: 0.8363 - val_loss: 0.5065 - learning_rate: 1.0000e-04
Epoch 21/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8576 - loss: 0.4123

155/155 ━━━━━━━━━━━━━━━━━━━━ 180s 1s/step - accuracy: 0.8576 - loss: 0.4122 - val_accuracy: 0.8390 - val_loss: 0.5003 - learning_rate: 1.0000e-04
Epoch 22/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8637 - loss: 0.3884

155/155 ━━━━━━━━━━━━━━━━━━━━ 178s 1s/step - accuracy: 0.8637 - loss: 0.3884 - val_accuracy: 0.8407 - val_loss: 0.4942 - learning_rate: 1.0000e-04
Epoch 23/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8666 - loss: 0.3788

155/155 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.8666 - loss: 0.3789 - val_accuracy: 0.8419 - val_loss: 0.4976 - learning_rate: 1.0000e-04
Epoch 24/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 180s 1s/step - accuracy: 0.8708 - loss: 0.3794 - val_accuracy: 0.8381 - val_loss: 0.5003 - learning_rate: 1.0000e-04
Epoch 25/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 177s 1s/step - accuracy: 0.8619 - loss: 0.3873 - val_accuracy: 0.8398 - val_loss: 0.5005 - learning_rate: 1.0000e-04
Epoch 26/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 179s 1s/step - accuracy: 0.8715 - loss: 0.3652 - val_accuracy: 0.8410 - val_loss: 0.4953 - learning_rate: 3.0000e-05
Epoch 27/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.8794 - loss: 0.3573 - val_accuracy: 0.8401 - val_loss: 0.4943 - learning_rate: 3.0000e-05
Epoch 28/30
155/155 ━━━━━━━━━━━━━━━━━━━━ 179s 1s/step - accuracy: 0.8815 - loss: 0.3487 - val_accuracy: 0.8395 - val_loss: 0.4935 - learning_rate: 3.0000e-05


## Image Prediction

1. Load the image from the given path.
2. Resize it to 224 × 224 pixels.
3. Convert the image to an array and normalize it.
4. Expand dimensions to match model input shape.
5. Make prediction using the trained model.
6. Reverse the class index to get the class name.
7. Display the predicted class label.

In [ ]:

import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "/content/drive/MyDrive/Butter-Noodles-S5.jpg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

index_to_class = {v: k for k, v in train_generator.class_indices.items()}
predicted_class = index_to_class[np.argmax(prediction)]

print("Predicted:", predicted_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Predicted: Noodles-Pasta


##Nutrition Database

1. Created a nutrition database using a Python dictionary.
2. Each food category contains values per 100 grams.
3. Stored nutritional information:
  - Calories
  - Protein
4. The dictionary is printed to display the stored data.

In [ ]:

nutri_per_100g = {
    "Bread":{"calories":265,"protein":9},
    "Dairy prodcut":{"calories":70,"protein":3.5},
    "Dessert":{"calories":400,"protein":4},
    "Egg":{"calories":155,"protein":13},
    "Fried food":{"calories":312,"protein":17},
    "Meat":{"calories":250,"protein":26},
    "Noodles-Pasta":{"calories":364,"protein":6},
    "Rice":{"calories":350,"protein":3.5},
    "Seafood":{"calories":206,"protein":22},
    "Soup":{"calories":150,"protein":2},
    "Vegetable-Fruit":{"calories":40,"protein":2}

}

print(nutri_per_100g)

{'Bread': {'calories': 265, 'protein': 9}, 'Dairy prodcut': {'calories': 70, 'protein': 3.5}, 'Dessert': {'calories': 400, 'protein': 4}, 'Egg': {'calories': 155, 'protein': 13}, 'Fried food': {'calories': 312, 'protein': 17}, 'Meat': {'calories': 250, 'protein': 26}, 'Noodles-Pasta': {'calories': 364, 'protein': 6}, 'Rice': {'calories': 350, 'protein': 3.5}, 'Seafood': {'calories': 206, 'protein': 22}, 'Soup': {'calories': 150, 'protein': 2}, 'Vegetable-Fruit': {'calories': 40, 'protein': 2}}


## Food Prediction with Nutrition Info

1. The model predicts the food with the highest probability.
2. The class index is reversed to get the food name.
3. he predicted food label is obtained.
4. Nutrition data **calories and protein per 100g** is got from the dictionary.
5. The food name and its nutrition values are displayed.

In [ ]:

prediction = model.predict(img_array)

index_to_class = {v: k for k, v in train_generator.class_indices.items()}
predicted_class = index_to_class[np.argmax(prediction)]

nutrition = nutri_per_100g[predicted_class]

print("Food:", predicted_class)
print("Calories per 100g:", nutrition["calories"], "kcal")
print("Protein per 100g:", nutrition["protein"], "g")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Food: Noodles-Pasta
Calories per 100g: 364 kcal
Protein per 100g: 6 g


##Nutrition Calculation Based on Weight

1. Takes user input weight in grams and converts it to float.
2. Calculates calories using:  
  **(calories per 100g × weight) ÷ 100**
3. Calculates protein using the same formula.
4. Formats output to 2 decimal places.
5. \n is used to print the result on a new line.

In [ ]:

weight = float(input("Enter weight in grams: "))

calories = (nutrition["calories"] * weight) / 100
protein = (nutrition["protein"] * weight) / 100

print(f"""
Food: {predicted_class}
Weight: {weight} g
Calories: {calories:.2f} kcal
Protein: {protein:.2f} g
""")

Enter weight in grams: 20

Food: Noodles-Pasta
Weight: 20.0 g
Calories: 72.80 kcal
Protein: 1.20 g



## Daily Calorie Calculation

1. This function calculates daily calorie needs.
2. Formula used:  
  **Daily Calories = Weight (kg) × 30**
3. It returns the estimated daily calorie requirement.

In [ ]:

def calculate_daily_calories(weight_kg):
    return weight_kg * 30

##Daily Protein Calculation

1. This function calculates daily protein requirement.
2. Assumes an average intake of 1.2g protein per kg body weight.
3. Formula used:  
  **Daily Protein = Weight (kg) × 1.2**
4. Returns the estimated daily protein need.

In [ ]:

def calculate_daily_protein(weight_kg):
    return weight_kg * 1.2

## User Daily Nutrition Estimation

1. Takes user body weight as input (in kg).
2. Calculates daily calorie needs using the calorie function.
3. Calculates daily protein needs using the protein function.
4. Displays the estimated daily calories and protein.
5. Output values are formatted without decimals.

In [ ]:
user_weight = float(input("Enter your body weight (kg): "))

daily_calorie_need = calculate_daily_calories(user_weight)
daily_protein_need = calculate_daily_protein(user_weight)

print(f"\nYour Estimated Daily Needs:")
print(f"Calories: {daily_calorie_need:.0f} kcal")
print(f"Protein: {daily_protein_need:.0f} g")

Enter your body weight (kg): 123

Your Estimated Daily Needs:
Calories: 3690 kcal
Protein: 148 g


In [19]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/best_food_model.h5")

In [20]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [21]:
validation_generator = val_generator
# Reset validation generator
validation_generator.reset()

# Predict
predictions = model.predict(validation_generator)

# Convert probabilities to predicted class index
y_pred = np.argmax(predictions, axis=1)

# True labels
y_true = validation_generator.classes

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


54/54 ━━━━━━━━━━━━━━━━━━━━ 903s 17s/step


##confusion Matrix Analysis

The confusion matrix shows strong diagonal dominance, indicating high classification accuracy. Most food categories are correctly predicted, with only minor misclassifications between visually similar classes. This confirms that the transfer learning model performs effectively for multi-class food image classification.

In [23]:
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[282   2  14  24  11  18   0   1   6   3   1]
 [  6  88  29   5   2   1   0   4   2   5   0]
 [ 14  10 395  15  12  26   0   3   9  11   5]
 [ 11   3  18 263  11  10   1   2   2   4   5]
 [ 10   2  13   7 276  13   2   0   3   1   0]
 [ 12   1  12  10  10 391   0   2   8   2   1]
 [  0   0   1   0   2   0 140   0   0   4   0]
 [  0   0   0   0   0   1   0  95   0   1   0]
 [  1   1  15  11   4  26   0   0 276   6   7]
 [  2   1   6   7   3   5   0   2   4 471   0]
 [  0   0   5   2   3   2   1   2   2   1 214]]


## Conclusion

In this project, we built a deep learning-based food classification system using transfer learning with MobileNetV2.  

The model successfully predicts food categories from images and provides nutritional information such as calories and protein per 100g.  

Additionally, the system calculates personalized daily calorie and protein requirements based on user body weight.  

The final model achieved an accuracy of **88%**, demonstrating good performance for food recognition and nutrition estimation.